# Gold Airline Monthly Fact Table

This notebook builds `gold.fact_airline_month` at an **airline × month** grain. It combines OPDI flight records with daily airport ATFM delays, estimates each airline’s delay exposure at the departure airport, and aggregates flight and delay measures by month.


## 1. Load Silver Data

Load the cleaned ATFM delay and OPDI flight tables from the Silver layer and inspect their schemas before transformation.


In [2]:
df_atfm_delay = spark.table("silver.atfm_delays")
df_opdi_flights = spark.table("silver.opdi_flights")

df_atfm_delay.printSchema()
print(df_atfm_delay.columns)
display(df_atfm_delay.head(10))

print("\n\n")

df_opdi_flights.printSchema()
print(df_opdi_flights.columns)
display(df_opdi_flights.head(10))


StatementMeta(, 6af2fe18-4148-4cb0-ac5b-7d4407bb2aea, 5, Finished, Available, Finished, False)

root
 |-- YEAR: integer (nullable = true)
 |-- MONTH_NUM: integer (nullable = true)
 |-- MONTH_MON: string (nullable = true)
 |-- FLT_DATE: date (nullable = true)
 |-- APT_ICAO: string (nullable = true)
 |-- APT_NAME: string (nullable = true)
 |-- STATE_NAME: string (nullable = true)
 |-- TOTAL_ARRIVALS: integer (nullable = true)
 |-- TOTAL_ATFM_DELAY_MINUTES: integer (nullable = true)
 |-- ACCIDENT_INCIDENT_DELAY_MINUTES: integer (nullable = true)
 |-- ATC_CAPACITY_DELAY_MINUTES: integer (nullable = true)
 |-- DEICING_DELAY_MINUTES: integer (nullable = true)
 |-- NON_ATC_EQUIPMENT_DELAY_MINUTES: integer (nullable = true)
 |-- AERODROME_CAPACITY_DELAY_MINUTES: integer (nullable = true)
 |-- ATC_INDUSTRIAL_ACTION_DELAY_MINUTES: integer (nullable = true)
 |-- AIRSPACE_MANAGEMENT_DELAY_MINUTES: integer (nullable = true)
 |-- NON_ATC_INDUSTRIAL_ACTION_DELAY_MINUTES: integer (nullable = true)
 |-- OTHER_DELAY_MINUTES: integer (nullable = true)
 |-- SPECIAL_EVENT_DELAY_MINUTES: integer (null

SynapseWidget(Synapse.DataFrame, 3bb690de-d994-4dc4-8685-91eb6a82dd0b)




root
 |-- icao_operator: string (nullable = true)
 |-- flight_id: long (nullable = true)
 |-- flight_date: date (nullable = true)
 |-- adep_icao: string (nullable = true)
 |-- ades_icao: string (nullable = true)
 |-- aircraft_type: string (nullable = true)
 |-- source_name: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- loaded_at: string (nullable = true)
 |-- year_month: integer (nullable = true)
 |-- carrier_type: string (nullable = true)
 |-- lcc_flag: integer (nullable = true)
 |-- network_flag: integer (nullable = true)
 |-- regional_flag: integer (nullable = true)
 |-- other_flag: integer (nullable = true)

['icao_operator', 'flight_id', 'flight_date', 'adep_icao', 'ades_icao', 'aircraft_type', 'source_name', 'batch_id', 'loaded_at', 'year_month', 'carrier_type', 'lcc_flag', 'network_flag', 'regional_flag', 'other_flag']


SynapseWidget(Synapse.DataFrame, 36b12c8f-eb8e-40e6-b302-7e8f3d08c248)

## 2. Prepare Daily ATFM Delays

Standardize the date and airport identifiers, then ensure the ATFM data has one row per airport and day. Calculate delay minutes per arrival for use as the flight-level delay exposure.


In [3]:
from pyspark.sql import functions as F

atfm_airport_day = (
    df_atfm_delay
    .filter(
        F.col("FLT_DATE").isNotNull() &
        F.col("APT_ICAO").isNotNull()
    )
    .groupBy(
        F.to_date("FLT_DATE").alias("flight_date"),
        F.upper(F.trim("APT_ICAO")).alias("airport_icao")
    )
    .agg(
        F.sum("TOTAL_ATFM_DELAY_MINUTES")
            .cast("double")
            .alias("airport_atfm_delay_minutes"),

        F.sum("TOTAL_ARRIVALS")
            .cast("long")
            .alias("airport_total_arrivals"),

        F.sum("ATFM_DELAYED_ARRIVALS")
            .cast("long")
            .alias("airport_delayed_arrivals"),

        F.sum("WEATHER_DELAY_MINUTES")
            .cast("double")
            .alias("airport_weather_delay_minutes")
    )
    .withColumn(
        "atfm_delay_minutes_per_flight",
        F.when(
            F.col("airport_total_arrivals") > 0,
            F.col("airport_atfm_delay_minutes") /
            F.col("airport_total_arrivals")
        )
    )
)

atfm_airport_day.printSchema()
print(atfm_airport_day.columns)
display(atfm_airport_day.head(10))

StatementMeta(, 6af2fe18-4148-4cb0-ac5b-7d4407bb2aea, 6, Finished, Available, Finished, False)

root
 |-- flight_date: date (nullable = true)
 |-- airport_icao: string (nullable = true)
 |-- airport_atfm_delay_minutes: double (nullable = true)
 |-- airport_total_arrivals: long (nullable = true)
 |-- airport_delayed_arrivals: long (nullable = true)
 |-- airport_weather_delay_minutes: double (nullable = true)
 |-- atfm_delay_minutes_per_flight: double (nullable = true)

['flight_date', 'airport_icao', 'airport_atfm_delay_minutes', 'airport_total_arrivals', 'airport_delayed_arrivals', 'airport_weather_delay_minutes', 'atfm_delay_minutes_per_flight']


SynapseWidget(Synapse.DataFrame, 19c7d3c6-6241-44ff-bdf1-a063aff0541e)

## 3. Prepare OPDI Flights

Select and standardize the required flight, airline, departure-airport, and carrier-classification fields. Flights without an airline code are assigned to an `UNKNOWN` member.


In [4]:
flights = (
    df_opdi_flights
    .select(
        F.col("flight_id"),
        F.to_date("flight_date").alias("flight_date"),
        F.upper(F.trim("icao_operator")).alias("icao_airline"),
        F.upper(F.trim("adep_icao")).alias("adep_icao"),
        F.col("carrier_type"),
        F.col("lcc_flag").cast("int"),
        F.col("network_flag").cast("int"),
        F.col("regional_flag").cast("int"),
        F.col("other_flag").cast("int")
    )
    .filter(
        F.col("flight_id").isNotNull() &
        F.col("flight_date").isNotNull()
    )
    .withColumn(
        "icao_airline",
        F.when(
            F.col("icao_airline").isNull() |
            (F.col("icao_airline") == ""),
            F.lit("UNKNOWN")
        ).otherwise(F.col("icao_airline"))
    )
)

flights.printSchema()
print(flights.columns)
display(flights.head(10))

StatementMeta(, 6af2fe18-4148-4cb0-ac5b-7d4407bb2aea, 7, Finished, Available, Finished, False)

root
 |-- flight_id: long (nullable = true)
 |-- flight_date: date (nullable = true)
 |-- icao_airline: string (nullable = true)
 |-- adep_icao: string (nullable = true)
 |-- carrier_type: string (nullable = true)
 |-- lcc_flag: integer (nullable = true)
 |-- network_flag: integer (nullable = true)
 |-- regional_flag: integer (nullable = true)
 |-- other_flag: integer (nullable = true)

['flight_id', 'flight_date', 'icao_airline', 'adep_icao', 'carrier_type', 'lcc_flag', 'network_flag', 'regional_flag', 'other_flag']


SynapseWidget(Synapse.DataFrame, 18e3150e-8c8c-45a1-96cc-b0fbea41ef4a)

## 4. Add ATFM Delay Exposure

Join each flight to the ATFM record for its departure airport and flight date. This assigns the airport’s estimated delay per arrival to the departing flight.


In [5]:
flights_with_atfm = (
    flights.alias("f")
    .join(
        atfm_airport_day.alias("a"),
        (
            (F.col("f.flight_date") == F.col("a.flight_date")) &
            (F.col("f.adep_icao") == F.col("a.airport_icao"))
        ),
        "left"
    )
    .select(
        "f.*",
        F.col("a.atfm_delay_minutes_per_flight")
            .alias("atfm_delay_experienced_minutes"),
        F.col("a.airport_atfm_delay_minutes"),
        F.col("a.airport_total_arrivals"),
        F.when(
            F.col("a.airport_icao").isNotNull(), 1
        ).otherwise(0).alias("atfm_match_flag")
    )
)


flights_with_atfm.printSchema()
print(flights_with_atfm.columns)
display(flights_with_atfm.head(10))


StatementMeta(, 6af2fe18-4148-4cb0-ac5b-7d4407bb2aea, 8, Finished, Available, Finished, False)

root
 |-- flight_id: long (nullable = true)
 |-- flight_date: date (nullable = true)
 |-- icao_airline: string (nullable = true)
 |-- adep_icao: string (nullable = true)
 |-- carrier_type: string (nullable = true)
 |-- lcc_flag: integer (nullable = true)
 |-- network_flag: integer (nullable = true)
 |-- regional_flag: integer (nullable = true)
 |-- other_flag: integer (nullable = true)
 |-- atfm_delay_experienced_minutes: double (nullable = true)
 |-- airport_atfm_delay_minutes: double (nullable = true)
 |-- airport_total_arrivals: long (nullable = true)
 |-- atfm_match_flag: integer (nullable = false)

['flight_id', 'flight_date', 'icao_airline', 'adep_icao', 'carrier_type', 'lcc_flag', 'network_flag', 'regional_flag', 'other_flag', 'atfm_delay_experienced_minutes', 'airport_atfm_delay_minutes', 'airport_total_arrivals', 'atfm_match_flag']


SynapseWidget(Synapse.DataFrame, 417ca696-1c10-4029-8074-5695681fe8b9)

## 5. Aggregate by Airline and Month

Group the enriched flights by airline and calendar month. Calculate flight counts, estimated ATFM delay exposure, airport coverage, and carrier-classification flags.


In [6]:
fact_airline_month = (
    flights_with_atfm
    .withColumn(
        "month_start_date",
        F.trunc("flight_date", "month").cast("date")
    )
    .groupBy(
        "icao_airline",
        "month_start_date"
    )
    .agg(
        F.countDistinct("flight_id").alias("flight_count"),

        F.sum("atfm_delay_experienced_minutes")
            .alias("atfm_delay_experienced_minutes"),

        F.avg("atfm_delay_experienced_minutes")
            .alias("avg_atfm_delay_experienced_minutes"),

        F.sum("atfm_match_flag").alias("atfm_matched_flight_count"),

        F.countDistinct(
            F.when(
                F.col("adep_icao").isNotNull() &
                (F.col("adep_icao") != ""),
                F.col("adep_icao")
            )
        ).alias("departure_airport_count"),

        F.max("lcc_flag").alias("lcc_flag"),
        F.max("network_flag").alias("network_flag"),
        F.max("regional_flag").alias("regional_flag"),
        F.max("other_flag").alias("other_flag")
    )
    .withColumn(
        "year",
        F.year("month_start_date")
    )
    .withColumn(
        "month_num",
        F.month("month_start_date")
    )
    .withColumn(
        "year_month",
        F.date_format("month_start_date", "yyyyMM").cast("int")
    )
    .withColumn(
        "atfm_match_rate",
        F.col("atfm_matched_flight_count") / F.col("flight_count")
    )
)

StatementMeta(, 6af2fe18-4148-4cb0-ac5b-7d4407bb2aea, 9, Finished, Available, Finished, False)

## 6. Add Airline Classification

Attach a single carrier type to each airline after aggregation. This preserves the required airline-by-month grain of the fact table.


In [7]:
airline_classification = (
    flights
    .filter(F.col("icao_airline") != "UNKNOWN")
    .groupBy("icao_airline")
    .agg(
        F.first("carrier_type", ignorenulls=True).alias("carrier_type")
    )
)

fact_airline_month = (
    fact_airline_month
    .join(airline_classification, "icao_airline", "left")
)

StatementMeta(, 6af2fe18-4148-4cb0-ac5b-7d4407bb2aea, 10, Finished, Available, Finished, False)

## 7. Validate the Gold Fact

Confirm that each airline and month combination is unique, the ATFM join has not duplicated flights, and the delay-match rate is reasonable.


In [8]:
duplicate_grain = (
    fact_airline_month
    .groupBy("icao_airline", "month_start_date")
    .count()
    .filter(F.col("count") > 1)
)

assert duplicate_grain.limit(1).count() == 0



source_flight_count = flights.select("flight_id").distinct().count()

joined_flight_count = (
    flights_with_atfm
    .select("flight_id")
    .distinct()
    .count()
)

assert source_flight_count == joined_flight_count

StatementMeta(, 6af2fe18-4148-4cb0-ac5b-7d4407bb2aea, 11, Finished, Available, Finished, False)

## 8. Publish the Gold Table

Write the validated result to `gold.fact_airline_month`, making it available for reporting and airline performance analysis.


In [9]:
# TODO - Publish it to the warehouse.

import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants

# Lakehouse
(
    fact_airline_month.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.fact_airline_month")
)

# Warehouse
(
    fact_airline_month.write
    .mode("overwrite")
    .synapsesql("EAA_Gold.dbo.fact_airline_month")
)

StatementMeta(, 6af2fe18-4148-4cb0-ac5b-7d4407bb2aea, 12, Finished, Available, Finished, False)